# SMS Spam Classifier

A Scikit-learn text classification pipeline using TF-IDF, moving from tabular data (Titanic project) into unstructured text. Includes a baseline Naive Bayes model and a tuned Logistic Regression model targeting recall on the minority class.

## Baseline Model

In [ ]:
# --- Baseline Model: Naive Bayes with TF-IDF ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv("spam.csv", encoding="latin-1")[["v1", "v2"]]
df.columns = ["label", "text"]

df["text"] = df["text"].fillna("")
df["label"] = df["label"].map({"ham": 0, "spam": 1})

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
preds = model.predict(X_test_tfidf)

print("Baseline Accuracy:", accuracy_score(y_test, preds))
print("Baseline F1-Score:", f1_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["ham", "spam"]))

## Tuned Model (Logistic Regression + class_weight balanced + GridSearchCV)

In [ ]:
# --- Tuned Model: Logistic Regression with class_weight balanced, GridSearchCV ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train_tfidf, y_train)

print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_
preds = best_model.predict(X_test_tfidf)

print("Tuned Accuracy:", accuracy_score(y_test, preds))
print("Tuned F1-Score:", f1_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["ham", "spam"]))